In [1]:
import os
import json
from pathlib import Path
import nbformat as nbf

out = Path("E:/Network_Theory_Class/NT_and_ABM/data/01_sample_selection_clean.ipynb")

nb = nbf.v4.new_notebook()
nb["metadata"] = {
    "kernelspec": {
        "display_name": "Python 3.12 (NT_and_ABM)",
        "language": "python",
        "name": "python3"
    },
    "language_info": {
        "name": "python",
        "version": "3.12"
    }
}

cells = []

cells.append(nbf.v4.new_markdown_cell(
"""# Meeting 1 — Select and prepare the master spatial cell dataset

This notebook creates the **fixed real-network node set** used for the course analyses.

**link:** https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE309522  
**Dataset:** NCBI GEO `GSE309522`  
**Sample:** `WT4` / `GSM9268674` — infected wild-type, 4 weeks post infection, replicate 4  
**Node definition:** one node = one Xenium-segmented cell  
**Final node set:** 10,000 spatially local WT4 cells

Selection rule:

1. Keep WT4 cells with `transcript_counts > 0`.
2. Compute the spatial median of all eligible WT4 cells.
3. Compute Euclidean distance from each cell centroid to that median.
4. Keep the 10,000 closest cells.

This rule is deterministic and reproducible. **Edges are not defined in this notebook**; network construction is handled separately.
"""
))

cells.append(nbf.v4.new_code_cell(
"""from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This notebook is expected to live in:
# coursework/meeting_1_networks_I/
ROOT = Path("../..").resolve()
DATA_ROOT = Path(
    os.environ.get(
        "NT_ABM_DATA_DIR",
        ROOT / "data"
    )
).resolve()

RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = RAW_DIR / "GSE309522_clusters_per_cell.csv.gz"
WT4_PATH = RAW_DIR / "GSM9268674_WT4_cells.csv.gz"

for path in (METADATA_PATH, WT4_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

print("Project root:", ROOT)
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## 1. Confirm the selected sample

The integrated `clusters_per_cell` table is used here only to summarize the WT replicates and document the sample-selection decision.

**Important:** its cell identifiers do not directly match the original WT4 Xenium `cells.csv.gz` identifiers, so it is **not** joined to the raw spatial cell table in this notebook.
"""
))

cells.append(nbf.v4.new_code_cell(
"""metadata = pd.read_csv(METADATA_PATH)

wt_summary = (
    metadata[metadata["sample_id"].str.startswith("WT")]
    .groupby("sample_id")
    .agg(
        n_cells=("cell_id", "size"),
        n_clusters=("seurat_cluster", "nunique"),
        median_counts=("nCount_Xenium", "median"),
        median_features=("nFeature_Xenium", "median"),
        group=("group", "first"),
        batch=("batch", "first"),
    )
    .sort_index()
)

display(wt_summary)
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""### Selected sample: WT4

WT4 was retained as the course master sample because it is a wild-type infected replicate with a manageable size in the integrated overview, contains all 11 Seurat clusters, and has intermediate transcript/feature counts relative to the other WT replicates.

The actual spatial network nodes are selected from the **original WT4 Xenium cell table** below.
"""
))

cells.append(nbf.v4.new_markdown_cell("## 2. Load the original WT4 Xenium cell table"))

cells.append(nbf.v4.new_code_cell(
"""wt4_cells = pd.read_csv(WT4_PATH)

print("WT4 raw shape:", wt4_cells.shape)
print("\\nColumns:")
print(wt4_cells.columns.tolist())

display(wt4_cells.head())
"""
))

cells.append(nbf.v4.new_code_cell(
"""qc_summary = pd.Series({
    "total_segmented_cells": len(wt4_cells),
    "cells_with_transcripts_gt_0": (wt4_cells["transcript_counts"] > 0).sum(),
    "cells_with_zero_transcripts": (wt4_cells["transcript_counts"] == 0).sum(),
})

display(qc_summary.to_frame("value"))
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## 3. Inspect the WT4 spatial distribution

`x_centroid` and `y_centroid` are the spatial coordinates used to define the real cell network.
"""
))

cells.append(nbf.v4.new_code_cell(
"""eligible = wt4_cells.loc[
    wt4_cells["transcript_counts"] > 0
].copy()

plt.figure(figsize=(10, 6))
plt.scatter(
    eligible["x_centroid"],
    eligible["y_centroid"],
    s=1,
    alpha=0.35,
)
plt.xlabel("x centroid")
plt.ylabel("y centroid")
plt.title("WT4 — spatial distribution of eligible Xenium cells")
plt.gca().set_aspect("equal")
plt.show()
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## 4. Select the fixed 10,000-cell master ROI

The spatial center is defined by the **median** x and y coordinates. The 10,000 eligible cells with the smallest Euclidean distance to this center are retained.

This preserves a contiguous local region and avoids random spatial subsampling.
"""
))

cells.append(nbf.v4.new_code_cell(
"""TARGET_N = 10_000

center_x = eligible["x_centroid"].median()
center_y = eligible["y_centroid"].median()

eligible["distance_to_center"] = np.hypot(
    eligible["x_centroid"] - center_x,
    eligible["y_centroid"] - center_y,
)

wt4_roi = (
    eligible
    .nsmallest(TARGET_N, "distance_to_center")
    .copy()
    .reset_index(drop=True)
)

assert len(wt4_roi) == TARGET_N
assert wt4_roi["cell_id"].is_unique
assert wt4_roi[["x_centroid", "y_centroid"]].notna().all().all()

print(f"Spatial center: ({center_x:.3f}, {center_y:.3f})")
print("ROI cells:", len(wt4_roi))
print("Maximum radius:", round(wt4_roi["distance_to_center"].max(), 3))
print(
    "x range:",
    round(wt4_roi["x_centroid"].min(), 3),
    "to",
    round(wt4_roi["x_centroid"].max(), 3),
)
print(
    "y range:",
    round(wt4_roi["y_centroid"].min(), 3),
    "to",
    round(wt4_roi["y_centroid"].max(), 3),
)
"""
))

cells.append(nbf.v4.new_code_cell(
"""plt.figure(figsize=(10, 6))

plt.scatter(
    eligible["x_centroid"],
    eligible["y_centroid"],
    s=1,
    alpha=0.12,
    label="Eligible WT4 cells",
)

plt.scatter(
    wt4_roi["x_centroid"],
    wt4_roi["y_centroid"],
    s=2,
    alpha=0.8,
    label="Selected 10,000-cell ROI",
)

plt.scatter(
    center_x,
    center_y,
    marker="x",
    s=80,
    label="Spatial median",
)

plt.xlabel("x centroid")
plt.ylabel("y centroid")
plt.title("WT4 — selected master ROI")
plt.gca().set_aspect("equal")
plt.legend()
plt.show()
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## 5. Save the master node table

The resulting CSV is the fixed node set for subsequent network construction and course analyses.

**Boundary note:** because this is a spatial ROI extracted from a larger tissue section, cells near the ROI boundary may have biological neighbors outside the selected region. This boundary effect should be acknowledged when interpreting local network measures.
"""
))

cells.append(nbf.v4.new_code_cell(
"""OUTPUT_PATH = PROCESSED_DIR / "WT4_master_roi_10000.csv"

wt4_roi.to_csv(OUTPUT_PATH, index=False)

check = pd.read_csv(OUTPUT_PATH)

assert check.shape[0] == TARGET_N
assert check["cell_id"].is_unique
assert check[["x_centroid", "y_centroid"]].notna().all().all()

print("Saved:", OUTPUT_PATH)
print("Saved shape:", check.shape)

display(check.head())
"""
))

cells.append(nbf.v4.new_markdown_cell(
"""## Final dataset definition

The course master real-network node set is now fixed as:

\[
V = \\text{10,000 WT4 Xenium cells closest to the WT4 spatial median}
\]

The next analysis step is to define the spatial **edge rule** and construct the network \(G=(V,E)\).
"""
))

nb["cells"] = cells

with out.open("w", encoding="utf-8") as f:
    nbf.write(nb, f)

print(f"Created: {out}")


<>:284: SyntaxWarning: invalid escape sequence '\['
<>:284: SyntaxWarning: invalid escape sequence '\['
C:\Users\Asus\AppData\Local\Temp\ipykernel_21116\3162671827.py:284: SyntaxWarning: invalid escape sequence '\['
  \[


Created: E:\Network_Theory_Class\NT_and_ABM\data\01_sample_selection_clean.ipynb
